<font size="6"><b>BIAS-VARIANCE TRADE-OFF</b></font>

<font size="5"><b>Serhat Çevikel</b></font>

In [ ]:
library(data.table)
library(tidyverse)
library(rlist)

In [ ]:
options(repr.matrix.max.rows=20, repr.matrix.max.cols=15) # for limiting the number of top and bottom rows of tables printed 

The bias–variance tradeoff describes the relationship between a model's complexity, the accuracy of its predictions, and how well it can make predictions on previously unseen data that were not used to train the model. In general, as the number of tunable parameters in a model increases, it becomes more flexible, and can better fit a training data set. That is, the model has lower error or lower bias. However, for more flexible models, there will tend to be greater variance to the model fit each time we take a set of samples to create a new training data set. 

(https://en.wikipedia.org/wiki/Bias%E2%80%93variance_tradeoff)

Let's remember that the true model that we try to estimate is:

${\displaystyle \mathbf {y} = \mathbf {X} {\boldsymbol {\beta }} + {\boldsymbol {\epsilon }}}$

While the estimated model is:

${\displaystyle \mathbf {y} = \mathbf {X} {\boldsymbol {\hat \beta }} + {\boldsymbol {e }}}$

The predictions estimated model makes are:

${\displaystyle \mathbf {\hat y} = \mathbf {X} {\boldsymbol {\hat \beta }}}$

And let's denote the predictions made by the true model as:

${\displaystyle \mathbf {\dot y} = \mathbf {X} {\boldsymbol {\beta }}}$

Let's assume there is some true model that maps a complex relationship between population predictors and the response. That may be because existing a multitude of variables or because the nature and shape of the bivariate relationships.

And let's assume that at any time we can only have a smaller sample of response and predictor values. We usually train the model and learn the parameters using a train sample, however, we assess the predictive power of the model on a test set, a set that is not used in training and so the model didn't see before.

We may opt for

- A simple model with fewer variables and/or incorporating simple type of (i.e. linear) relationships
- A complex model with more variables and/or incorporating more complex type of (i.e., polynomial, power or interactive) relationships.

The continuum between the simple and complex models present a very important trade-off in statistical learning, Bias-Variance trade-off.

When we run the model on different train samples and assess the predictive power on a certain test set and calculate prediced values $\hat y$:

- *Variance* refers to the amount by which $\hat y$ would change when the model is estimated on different train samples.
- *Bias* refers to the error that is introduced by trying to estimate the true model with a simpler model, so the difference between $\hat y$ and $\dot y$

For a given unseen values of the predictors $x_0$:

- The response value is $y_0$
- The prediction by the estimated model is $\hat{y}_0$
- The residual is $e_0 = y_0 - \hat{y}_0$
- The prediction by the true model is $\dot{y}_0$
- The bias is $\text{Bias}_0 = \hat{y}_0 - \dot{y}_0$
- The error is $\epsilon_0 = y_0 - \dot{y}_0$

The bias-variance trade-off posits that:

$\operatorname E(e_0^2) = \operatorname {Var}(\hat{y}_0) + \operatorname E({\text{Bias}_0})^2 + {\epsilon_0}^2$

So the expected test MSE is equal to the variance of predictions of estimated models, average squared biases between predictions of the estimated and true models and the squared value of errors in the true model.

# Decomposition simulation

We will demonstrate the bias-variance tradeoff visually using simulation.

This section borrow heavily from: `4.5 Using Simulation to Estimate Bias and Variance` section of a great source book *Basics of Statistical Learning* by David Dalpiaz.

(https://statisticallearning.org/bias-variance-tradeoff.html#using-simulation-to-estimate-bias-and-variance)

First, we will create a data simulator function that takes as arguments:
- A function that defines the relationship between the predictor and response
- Number of observationss
- Standard deviation of the error term

The function draws random samples from standard uniform distribution, calculates the true y values and draws normally distributed error terms and adds to the true y values for the observed y values.  

In [ ]:
simdat <- function(func, sizex, ersd = 0.3)
{
    datx <- data.table(x = runif(sizex))
    datx[, ytrue := func(x)]
    datx[, erx := rnorm(sizex, 0, ersd)]
    datx[, y := ytrue + erx]
    datx[]
}

The function that models the true relationship between the predictor and response is below, that sums the predictor value and its square:

In [ ]:
squarex <- function(x) rowSums(outer(x, 1:2, "^"))

Now we will create a function to run a linear regression model that takes the relationship between response and predictor as the defined polynomial degrees of the predictor.

Values for different polynomial degrees are orthagonalized by `poly` function, so there is no collinearity among degrees.

When the degree is 0, the model is basically an intercept only or null model.

In [ ]:
lmp <- function(deg, dat)
{
    if (deg == 0)
    {
        lm(y ~ 1, dat)
    } else
    {
    lm(y ~ poly(x, deg), dat)
    }
}

We also create a function that runs multiple models with varying complexity on the same train data and creates the predictions of those models on the selected fix predictor value:

In [ ]:
batchpred <- function(simd, testd)
{
    models <- lapply(degrees, lmp, simd)
    sapply(models, function(x) predict(x, testd))
}

We will define some hyper-parameter values:

- nsim: Number of different train datasets
- ersdx: Standard distribution of the error terms
- degrees: Polynomial degrees for the models with varying complexity
- funcx: The function that models the true relationship between response and predictor
- simx: The selected fix predictor value to get the predictions of the models

In [ ]:
nsim <- 1e3 # 500
ersdx <- 0.1
degrees <- 1:9
funcx <- squarex #fourthx
simx <- 0.5

We create the simulated datasets into a list:

In [ ]:
set.seed(10)
simdtl <- replicate(nsim, simdat(funcx, 100, ersdx), simplify = F)

We calculate the true y value for the selected predictor value using the function for the true model:

In [ ]:
testdt <- data.table(x = rep(simx, nsim))
testdt[, ytrue := funcx(x)]

And add a random error term to that data point for each simulation run:

In [ ]:
set.seed(30)
testdt[, erx := rnorm(nsim, 0, ersdx)]
testdt[, y := ytrue + erx]

Run the models with different complexities (polynomial degrees) and calculate the predicted y values for the selected predictor value (only one row suffices since the predictor and true y values are the same for all rows in testdt). The output will be a list of predicted value vectors:

In [ ]:
allpreds <- lapply(simdtl, batchpred, testdt[1])

Combine all of them into a single matrix:

In [ ]:
allpredsm <- do.call(rbind, allpreds)
colnames(allpredsm) <- degrees

And into a data table:

In [ ]:
allpreds_dt <- mapply(function(x, y) data.table(pred = x, deg = y),
       as.list(as.data.frame(allpredsm)),
       degrees, SIMPLIFY = F) %>% rbindlist
#allpreds_dt[, deg := factor(deg, ordered = T)]        
allpreds_dt[, deg := as.integer(deg)]        

We get the true y value for the test, given the true model between the predictor and response:

In [ ]:
truthx <- testdt[1, ytrue]
truthx

And get the observed y values in the test data:

In [ ]:
ysim <- testdt$y

And we add those observed y values of the test data to our data.table with predictions:

In [ ]:
allpreds_dt[, yval := rep(ysim, length.out = .N)]

In [ ]:
allpreds_dt %>%
mutate_at("deg", factor, ordered = T) %>%
ggplot(aes(x = deg, y = pred, color = deg)) +
geom_jitter() +
geom_boxplot() +
geom_hline(yintercept = truthx, color = "red")

Now we create three functions for:

- The estimated mean squared error, total error that we want to decompose
- Bias component, the mean of the difference between the predictions of the model and the true y values, predictions of the true model
- Variance component, the variance of the estimates

We will also add the mean squared valuess of the error term in true model (diffrence between the observed y value and the true y value in test data) as the last component.

In [ ]:
mse <- function(observed, estimate) mean((estimate - observed)^2)
bias <- function(estimate, truth) mean(estimate - truth)
varr <- function(estimate) sum((estimate - mean(estimate))^2)/(length(estimate) - 1)

In [ ]:
bvto <- allpreds_dt[, .(mses = mse(yval, pred),
                biasx = bias(pred, truthx)^2,
                varx = varr(pred),
                sqer = mean((yval - truthx)^2)),
            by = deg]

In [ ]:
bvto

We can confirm that the decomposition holds:

In [ ]:
round(bvto[, (mses - (biasx + varx + sqer)) / mses], 1)

We can show the bias-variance tradeoff with a line chart:

In [ ]:
bvto %>%
pivot_longer(-"deg") %>%
#filter(name == "biasx") %>%
ggplot(aes(x = deg, y = value, color = name, group = name)) +
geom_line() +
scale_x_discrete()

Or an area chart:

In [ ]:
mses_dt <- bvto %>%
pivot_longer(-"deg") %>%
filter(name == "mses")

In [ ]:
bvto %>%
pivot_longer(-"deg") %>%
filter(name != "mses") %>%
mutate_at("name", factor, levels = c("biasx", "varx", "sqer")) %>% 
ggplot(aes(x = deg, y = value, fill = name, group = name)) +
geom_area() +
geom_line(data = mses_dt,
          aes(x = deg, y = value, group = name), color = "black") +
scale_x_discrete()

Here, the squared error value is the same across all estimated models, since it is basically the mean squared difference between the observed value and the predicted value of the TRUE model, not the estimated HAT models.

Variance increases as the model complexity is increased: The variance of the predictions of different models of the same complexity for the same predictor value has a larger dispersion.

Bias decreases with higher model complexity: While the variance is higher, on the average the difference between the predictions of the estimated model and the true model is closer.

With increasing model complexity, total MSE - mean squared difference between the predictions of estimated models and observed y values - first decreases with lower bias, then starts to rise due to increased variance of predictions.

# Train and test errors vs complexity

Now instead of checking the bias-variance decomposition of the predictions of the models on the same predictor value, we will create a second collection of simulated datasets for test purposes.

For each complexity levels, the models will be estimated using a train set and tested on a test set. We will collect the MSE values for each train and test sets.

First of all, this the MSE value will be calculated for each model and its corresponding dataset, not across the selected predicted values of all models. So the degrees of freedom in the denominator is corrected by subtracting the number of estimated parameters.

In [ ]:
mse2 <- function(truth, estimate, npar)
{
    sum((estimate - truth)^2) / (length(estimate) - npar)
}

Batch function for predictions is also changed so that, for each pair of train and test sets a list containing the train and test predictions for all estimated models with different complexities are returned:

In [ ]:
batchpred2 <- function(simd, testd)
{
    models <- lapply(degrees, lmp, simd)
    trainpred <- lapply(models, function(x) predict(x, simd))
    names(trainpred) <- degrees
    testpred <- lapply(models, function(x) predict(x, testd))
    names(testpred) <- degrees
    list(trainp = trainpred, testp = testpred)
}

Now we simulate the test datasets:

In [ ]:
set.seed(20)
simdtl2 <- replicate(nsim, simdat(funcx, 100, ersdx), simplify = F)

We estimate the models and collect train and test predictions:

In [ ]:
traintestl <- mapply(batchpred2, simdtl, simdtl2, SIMPLIFY = F)

With the below steps, we collect and arrange all train and test predictions for all models and all simulation runs into a tidy and long format data.table:

In [ ]:
trainl <- traintestl %>% list.select(trainp) %>%
    list_flatten() %>% lapply(as.data.table)

testl <- traintestl %>% list.select(testp) %>%
    list_flatten() %>% lapply(as.data.table)

train_dt <- mapply(function(x, y, z) x[, (c("yval", "iter", "setx")) := .(y$y, z, "train")],
       trainl, simdtl, seq_along(trainl),
        SIMPLIFY = F) %>% rbindlist

test_dt <- mapply(function(x, y, z) x[, (c("yval", "iter", "setx")) := .(y$y, z, "test")],
       testl, simdtl2, seq_along(testl),
        SIMPLIFY = F) %>% rbindlist

traintest_dt <- rbind(train_dt, test_dt)

traintest_dt2 <-  traintest_dt %>% pivot_longer(cols = as.character(degrees),
                             names_to = "model",
                             values_to = "ypred")

setDT(traintest_dt2)
traintest_dt2[, model := as.integer(model)]

In [ ]:
str(traintest_dt2)

And we calculate the mse values for each set of predictions:

In [ ]:
traintest_mse <- traintest_dt2[, .(mses = mse2(yval, ypred, model + 1)),
              by = c("iter", "setx", "model")]

We first create the boxplots for test and train MSE values acrosss simulated sets for each model complexity:

In [ ]:
traintest_mse %>%
ggplot(aes(x = mses, color = setx)) +
geom_boxplot() +
xlim(0, 0.02) + 
facet_wrap(~ model, ncol = 1)

The overall distribution of train MSE values are always lower than that of the test MSE values.

Train MSE values always decrease with higher model complexity albeit at a lower pace.

Test MSE values first level off and then start to increase again with model complexity.

We can also show the the trajectory of the means of the MSE values across all simulated datasets:

In [ ]:
traintest_mse2 <- traintest_mse[, .(mses = mean(mses)), by =
                                c("setx", "model")]

In [ ]:
traintest_mse2 %>%
mutate_at("model", factor, ordered = T) %>%
ggplot(aes(x = model, y = mses, color = setx, group = setx)) +
geom_line()

The above interpretation is confirmed. In this example there is an optimal level of model complexity (here in two degree polynomials) so that train MSE is sufficiently low and test MSE is at a minimum.

# Resources on Bias-Variance Trade-Off

- James et al., 2023 (Corrected printing), An Introduction to Statistical Learning, Section 2.2.2
- Dalpiaz, 2020, Basics of Statistical Learning, Chapter 4 (https://statisticallearning.org/bias-variance-tradeoff.html)